# NB06 — UAV verification v2: six Nadine circles, 68 source polygons

Prepare clean verification assets for **circles 8, 12, 13, 16, 21, 33** using the **updated** `Data/Annotations/Verification/Verification.shp` and the original `Circles_ROI_NAdine.shp`.

**Mandatory QA:**
- Check there are **68 raw verification features**; verify native coordinate systems and each circle's UAV RGB mosaic by geometry.
- Repair invalid polygon geometries **in memory only**, stopping if repairs change area by more than 1%.
- Determine circle membership using a polygon representative point, and measure how much of each polygon lies inside its assigned circle.
- Detect **exact/near-exact duplicates** by polygon intersection-over-union (IoU ≥ 0.9999); retain the earliest source row as the canonical ID. Report strong partial overlaps separately (IoU ≥ 0.90), without silently deleting them.
- Produce a **unique-only** GEE table plus a complete **68-row mapping CSV**, preserving all source IDs and duplicate aliases.
- Generate **six fresh** high-resolution RGB clips with an 8 m viewing buffer and tiled compressed GeoTIFFs. No raster mosaicking between unrelated sites is needed; each clip comes from its corresponding orthomosaic.

**Safety:** source shapefiles and original UAV mosaics are never edited. Outputs go to the versioned directory `Outputs/V3/06_GEE_Verification_v2`; old GEE asset names are **not overwritten automatically**. Review the QA table and duplicates **before** enabling export. GeoTIFFs and zipped shapefiles can be uploaded manually in GEE's Asset Manager.

**Identifier convention:** each source feature receives `C08_V00000`, etc., based on its *original zero-based shapefile row*. GEE table uses a short `verify_id` attribute to avoid the shapefile 10-character field-name limit. The companion CSV records any original ID → canonical ID mapping.


## 01 — Read the six circles, updated annotations and orthomosaics

In [ ]:
%pip -q install geopandas pyogrio rasterio shapely pandas matplotlib
from pathlib import Path
from collections import defaultdict
import json, math, os, tempfile, zipfile, hashlib, time
import pandas as pd, numpy as np, geopandas as gpd, rasterio
from shapely.geometry import box, mapping
from shapely import make_valid
from shapely.ops import unary_union
from rasterio.mask import mask
from rasterio.enums import Resampling
from google.colab import drive
from IPython.display import display

if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
ANN=ROOT/'Data/Annotations/Verification/Verification.shp'
CIRCLE_FILES=list((ROOT/'Data').rglob('Circles_ROI_NAdine.shp'))
RGB_DIR=ROOT/'Data/Ortho_RGB'
OUT=ROOT/'Outputs/V3/06_GEE_Verification_v2'
OUT.mkdir(parents=True,exist_ok=True)
CIRCLE_IDS=[8,12,13,16,21,33]
EXPECTED_SOURCE_COUNT=68
BUFFER_M=8.0
DUPLICATE_IOU=0.9999
REVIEW_IOU=0.90
assert ANN.is_file(),ANN
assert len(CIRCLE_FILES)==1,CIRCLE_FILES
assert RGB_DIR.is_dir()
print('Verification source:',ANN, 'size:',ANN.stat().st_size)
print('Nadine circles:',CIRCLE_FILES[0])

raw=gpd.read_file(ANN)
all_circles=gpd.read_file(CIRCLE_FILES[0])
assert raw.crs is not None and all_circles.crs is not None
assert len(raw)==EXPECTED_SOURCE_COUNT,(
    f'Expected {EXPECTED_SOURCE_COUNT} UPDATED polygons; got {len(raw)}. '
    'Check Drive sync, source shapefile and existing Colab variables.')
assert all_circles.shape[0]>max(CIRCLE_IDS)
raw=raw.reset_index(drop=True)
raw['source_row']=np.arange(len(raw),dtype=int)
circles=all_circles.iloc[CIRCLE_IDS].copy()
circles['circle_id']=CIRCLE_IDS   # Zero-based source row numbers, consistent with NB01.
print('New verification count:',len(raw),'CRS:',raw.crs,'fields:',list(raw.columns))
print('Circle count:',len(circles),'CRS:',circles.crs)
print('Circle IDs:',circles.circle_id.tolist())
print('Available RGB mosaics:',[p.name for p in sorted(RGB_DIR.glob('*.tif'))])


## 02 — Geometry repair, membership and automatic UAV RGB mapping

**Stop at this stage if the mapping or circle/polygon counts look wrong.** Source feature IDs derive from the *full shapefile row order*, not from the number of features selected in a circle.


In [ ]:
def repair(g,label,area_change_limit=0.01):
    if g is None or g.is_empty:raise ValueError(f'{label}: null/empty geometry')
    if g.is_valid and g.geom_type in ('Polygon','MultiPolygon'):return g
    fixed=make_valid(g);pieces=[]
    def visit(q):
        if q.geom_type=='Polygon':pieces.append(q)
        elif hasattr(q,'geoms'):
            for part in q.geoms:visit(part)
    visit(fixed)
    if not pieces:raise ValueError(f'{label}: no polygon after repair')
    fixed=unary_union(pieces)
    change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-9)
    if not fixed.is_valid or change>area_change_limit:
        raise ValueError(f'{label}: unsafe geometry repair, area change {change:.3%}')
    print('REPAIRED',label,'area change',f'{change:.4%}')
    return fixed

raw.geometry=[repair(g,f'verification row {i}') for i,g in enumerate(raw.geometry)]
circles.geometry=[repair(g,f'circle {cid}') for cid,g in zip(circles.circle_id,circles.geometry)]
# Compare areas in metric UTM CRS even when source CRS differs.
METRIC='EPSG:32635'
v=raw.to_crs(METRIC).copy()
cs=circles.to_crs(METRIC).copy()
circle_dict={int(row.circle_id):row.geometry for row in cs.itertuples()}
assignment=[];inside_fraction=[]
for row in v.itertuples():
    hit=[cid for cid,g in circle_dict.items() if g.covers(row.geometry.representative_point())]
    if len(hit)!=1:
        raise ValueError(f'Row {row.source_row}: representative point belongs to {hit}; '
                         'cannot uniquely assign circle.')
    cid=hit[0]; assignment.append(cid)
    frac=row.geometry.intersection(circle_dict[cid]).area/max(row.geometry.area,1e-9)
    inside_fraction.append(frac)
v['circle_id']=assignment
v['inside_fraction']=inside_fraction
v['verify_id']=[f'C{int(cid):02d}_V{int(i):05d}' for i,cid in zip(v.source_row,v.circle_id)]
assert v.verify_id.is_unique
assert set(v.circle_id)==set(CIRCLE_IDS),(
    'At least one requested circle has no verification polygons',v.groupby('circle_id').size().to_dict())
print('Polygons per circle:')
display(v.groupby('circle_id').agg(polygons=('verify_id','size'),
      minimum_inside_fraction=('inside_fraction','min')).round(4))
boundary_warnings=v[v.inside_fraction<0.99]
if len(boundary_warnings):
    print('WARNING: polygons crossing outside circle. Inspect before exporting:')
    display(boundary_warnings[['verify_id','circle_id','inside_fraction']])
else:
    print('All polygons fall ≥99% inside assigned circle.')

# Match every requested circle to exactly one RGB orthomosaic geographically.
rgb_files=sorted(p for p in RGB_DIR.glob('*.tif')
                 if not any(x in p.stem.lower() for x in ('dsm','ms_')))
site_rgb={}
for cid,g in circle_dict.items():
    matches=[]
    for p in rgb_files:
        with rasterio.open(p) as ds:
            if ds.count<3 or ds.crs is None:continue
            geom=gpd.GeoSeries([g],crs=METRIC).to_crs(ds.crs).iloc[0]
            frac=geom.intersection(box(*ds.bounds)).area/max(geom.area,1e-9)
            if frac>0.95:matches.append((p,frac))
    if len(matches)!=1:
        raise ValueError(f'Circle {cid}: expected one RGB mosaic, got '+
                         str([(p.name,round(fr,3)) for p,fr in matches]))
    site_rgb[cid]=matches[0][0]
print('CIRCLE → RGB MAPPING')
for cid,p in sorted(site_rgb.items()):print(f'  {cid:2d}: {p.name}')


## 03 — Exhaustive duplicate and overlap audit (mandatory)

**Exact duplicates** are detected using geometric overlap, *not ID or centroid alone*. The later record in each exact-duplicate pair becomes an alias of the first. Any high but nonexact IoU remains a distinct polygon and is listed for inspection. No source file is edited.

This stage saves a complete source-to-canonical ID mapping and audit, but **does not upload to Earth Engine**.


In [ ]:
from shapely.errors import GEOSException
def safe_intersection_area(a,b):
    try:return a.intersection(b).area
    except GEOSException:
        from shapely import set_precision
        return set_precision(a,0.001).intersection(set_precision(b,0.001)).area

parents={id:id for id in v.verify_id}
def root(id):
    while parents[id]!=id:
        id=parents[id]
    return id
def combine(a,b):
    a=root(a);b=root(b)
    if a!=b:parents[b]=a  # retain first encountered row

pairs=[]
features=list(v.itertuples())
for i,a in enumerate(features):
    for b in features[i+1:]:
        if a.circle_id!=b.circle_id:continue
        if not a.geometry.intersects(b.geometry):continue
        ia=safe_intersection_area(a.geometry,b.geometry)
        if ia<=0:continue
        aa=a.geometry.area;ab=b.geometry.area
        union=aa+ab-ia
        iou=ia/max(union,1e-12)
        containment=ia/max(min(aa,ab),1e-12)
        if iou>=REVIEW_IOU or containment>=REVIEW_IOU:
            pairs.append(dict(id_1=a.verify_id,id_2=b.verify_id,circle_id=int(a.circle_id),
                              iou=float(iou),containment=float(containment),
                              centroid_distance_m=float(a.geometry.centroid.distance(b.geometry.centroid)),
                              identical=bool(iou>=DUPLICATE_IOU)))
        if iou>=DUPLICATE_IOU:
            combine(a.verify_id,b.verify_id)
audit=pd.DataFrame(pairs,columns=['id_1','id_2','circle_id','iou','containment',
                                  'centroid_distance_m','identical'])
if len(audit):
    print('POSSIBLE OVERLAPS / EXACT DUPLICATES:')
    display(audit.round(5))
else:print('No strong overlaps (IoU or containment ≥0.90).')
audit.to_csv(OUT/'verification_overlap_audit.csv',index=False)
v['canonical_id']=[root(id) for id in v.verify_id]
v['is_duplicate']=(v.verify_id!=v.canonical_id)
canonical=v[~v.is_duplicate].copy()
assert len(canonical)+int(v.is_duplicate.sum())==EXPECTED_SOURCE_COUNT
print('RAW features:',len(v),'| exact duplicate aliases:',int(v.is_duplicate.sum()),
      '| UNIQUE verification polygons:',len(canonical))
print('UNIQUE count by circle:')
display(canonical.groupby('circle_id').size().rename('unique_polygons').to_frame())
strong_not_exact=audit[(audit.iou< DUPLICATE_IOU) & (audit.iou>=REVIEW_IOU)] if len(audit) else audit
if len(strong_not_exact):
    print('WARNING: strong but NONIDENTICAL polygon overlaps require visual inspection.')
    display(strong_not_exact)
# Preserve all 68 rows and metadata: this is the main scientific provenance table.
cols=['verify_id','circle_id','source_row','canonical_id','is_duplicate','inside_fraction']
v[cols].to_csv(OUT/'verification_68_source_mapping.csv',index=False)
canonical[['verify_id','circle_id','source_row','inside_fraction']].to_csv(
    OUT/'verification_unique_index.csv',index=False)
print('Audit outputs written to',OUT)


## 04 — Export clean vector assets and six clipped RGB GeoTIFFs

**Enable only after inspecting Sections 02–03.** Output names use the original conventions and contain all six circles; use the versioned `OUT` folder, not the old NB06 output folder.

The GEE table stores **unique polygons only** and uses field names `verify_id`, `circle_id`, `source_row` to survive zipped shapefile conversion. A separate 68-record mapping CSV documents original duplicate aliases. Images are cropped from existing orthomosaics using an 8 m viewing buffer around the union of each circle and its full verification crowns and their original resolution.

**Repair of the mislocated circle 33 Earth Engine image (2026-10-21):** cell 04 now has `REBUILD_ONLY_CIRCLE_33=True`. This explicitly rebuilds just the clipped 33 GeoTIFF from its geographically matched source, verifies >=95% spatial circle coverage and >=50% non-nodata pixels, and prints the expected WGS84 bounding box. **The existing Earth Engine asset is not modified by Colab:** manually replace/re-upload `circle_33_UAV_RGB.tif` to the GEE asset `projects/musanga-511110/assets/Verification/circle_33_UAV_RGB`, then rerun the GEE coverage diagnostic. Other five RGB assets remain unchanged.


In [ ]:
RUN_EXPORT=False                   # Full six-circle publication (normally leave False)
REBUILD_ONLY_CIRCLE_33=True         # Targeted repair: regenerates & audits circle 33 only
ALLOW_BOUNDARY_WARNINGS=False
ALLOW_STRONG_NONEXACT_OVERLAPS=False

def zipped_shape(gdf,name):
    destination=OUT/f'{name}.zip'
    with tempfile.TemporaryDirectory() as temp:
        temp=Path(temp);shp=temp/f'{name}.shp'
        gdf.to_file(shp,driver='ESRI Shapefile',index=False)
        with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_DEFLATED) as zf:
            for f in sorted(temp.glob(name+'.*')):zf.write(f,arcname=f.name)
    print('TABLE READY:',destination.name,'features:',len(gdf),flush=True)
    return destination

def prepare_vectors():
    # GEE ingestion of zipped shapefiles is most reliable in EPSG:4326.
    polygons=canonical[['verify_id','circle_id','source_row','geometry']].to_crs('EPSG:4326').copy()
    circ=cs[['circle_id','geometry']].to_crs('EPSG:4326').copy()
    polygons.to_file(OUT/'verification_unique.geojson',driver='GeoJSON')
    circ.to_file(OUT/'circles_6.geojson',driver='GeoJSON')
    zipped_shape(polygons,'verification_unique')
    zipped_shape(circ,'circles_6')

def verify_rgb_coverage(destination,cid,required_coverage=0.95):
    """Fail if the saved RGB footprint is not spatially aligned with its circle.

    Verify real non-nodata RGB pixels too, not only TIFF metadata. Return
    coverage and valid fraction for traceable GEE upload diagnostics.
    """
    circle_geom=circle_dict[int(cid)]
    with rasterio.open(destination) as ds:
        assert ds.count>=3 and ds.crs is not None,(destination,'Not a 3-band georeferenced RGB')
        native=gpd.GeoSeries([circle_geom],crs=METRIC).to_crs(ds.crs).iloc[0]
        footprint=box(*ds.bounds)
        coverage=native.intersection(footprint).area/max(native.area,1e-12)
        assert coverage>=required_coverage,(
            f'Circle {cid}: WRONG FOOTPRINT in {destination.name}; '
            f'only {coverage:.1%} of circle covered (needs {required_coverage:.0%}). '
            f'Raster extent {ds.bounds}.')
        from rasterio.windows import from_bounds
        from rasterio.features import geometry_mask
        bounds=native.bounds
        win=from_bounds(*bounds,transform=ds.transform).round_offsets().round_lengths()
        from rasterio.windows import Window
        win=win.intersection(Window(0,0,ds.width,ds.height))
        h,w=int(win.height),int(win.width)
        assert h>0 and w>0,(cid,'Empty raster/circle intersection')
        stride=max(1,math.ceil(max(h,w)/512))
        sampled_h=max(1,math.ceil(h/stride))
        sampled_w=max(1,math.ceil(w/stride))
        mask_values=ds.dataset_mask(window=win,out_shape=(sampled_h,sampled_w),
                                    resampling=Resampling.nearest)>0
        inside=geometry_mask([mapping(native)],out_shape=(sampled_h,sampled_w),
           transform=ds.window_transform(win)*rasterio.Affine.scale(
               w/sampled_w,h/sampled_h),invert=True)
        count=int(inside.sum())
        assert count>0,(cid,'No sampled pixels inside circle')
        valid_fraction=float((mask_values&inside).sum()/count)
        assert valid_fraction>=0.50,(
            f'Circle {cid}: RGB footprint overlaps but only {valid_fraction:.1%} '
            'valid/non-nodata pixels. Verify clip, source image mask and nodata.')
        bounds_wgs=gpd.GeoSeries([footprint],crs=ds.crs).to_crs('EPSG:4326').total_bounds
        return dict(circle_id=int(cid),coverage_fraction=float(coverage),
                    valid_fraction=valid_fraction,source_crs=str(ds.crs),
                    raster_bounds_wgs84=list(map(float,bounds_wgs)),
                    width=int(ds.width),height=int(ds.height),
                    pixel_size=list(map(float,ds.res)))

def prepare_rgb(target_ids=None,force_rebuild_ids=None):
    target_ids=list(CIRCLE_IDS if target_ids is None else target_ids)
    force_rebuild_ids=set([] if force_rebuild_ids is None else force_rebuild_ids)
    assert set(target_ids).issubset(CIRCLE_IDS)
    results=[]
    # Keep successful outputs, but detect and rebuild incomplete TIFFs.
    # Crop must include complete verification crowns, including outside circle.
    for cid in target_ids:
        p=site_rgb[cid]
        destination=OUT/f'circle_{cid:02d}_UAV_RGB.tif'
        polys=v.loc[v.circle_id==cid,'geometry']
        crop_geom=circle_dict[cid].union(unary_union(list(polys))).buffer(BUFFER_M)
        if destination.exists() and cid not in force_rebuild_ids:
            try:
                with rasterio.open(destination) as existing:
                    good=(existing.count==3 and existing.crs is not None
                          and existing.width>0 and existing.height>0
                          and existing.is_tiled
                          and existing.block_shapes[0][0]%16==0
                          and existing.block_shapes[0][1]%16==0)
                    if good:
                        check_geom=gpd.GeoSeries([crop_geom],crs=METRIC).to_crs(existing.crs).iloc[0]
                        good=box(*existing.bounds).buffer(max(existing.res)*2).covers(check_geom)
                        if good:
                            verify_rgb_coverage(destination,cid)
                if good:
                    diag=verify_rgb_coverage(destination,cid)
                    results.append(dict(source_ortho=p.name,clip_file=destination.name,**diag))
                    print('REUSE spatially VERIFIED TIFF:',destination.name,diag,flush=True)
                    continue
            except Exception as exc:
                print('Existing TIFF unreadable:',destination.name,str(exc)[:180],flush=True)
            print('REBUILD incomplete/incorrect TIFF:',destination.name,flush=True)
            # Do not delete the previously generated file until a new one is verified.
        temp=destination.with_name(destination.stem+'.writing.tif')
        if temp.exists():temp.unlink()
        try:
            with rasterio.open(p) as src:
                geom=gpd.GeoSeries([crop_geom],crs=METRIC).to_crs(src.crs).iloc[0]
                data,transform=mask(src,[mapping(geom)],crop=True,filled=True,
                     nodata=src.nodata if src.nodata is not None else 0,indexes=[1,2,3])
                profile=src.profile.copy()
                # Source block sizes may be noncompliant when writing tiled output.
                for key in ('blockxsize','blockysize','photometric','nbits'):
                    profile.pop(key,None)
                profile.update(driver='GTiff',count=3,width=data.shape[2],height=data.shape[1],
                               transform=transform,tiled=True,blockxsize=256,blockysize=256,
                               compress='DEFLATE',BIGTIFF='IF_SAFER',interleave='pixel')
                with rasterio.open(temp,'w',**profile) as dst:
                    dst.write(data)
            with rasterio.open(temp) as check:
                assert check.count==3 and check.crs is not None
                assert check.width>0 and check.height>0
                assert check.is_tiled and check.block_shapes[0]==(256,256)
            diag=verify_rgb_coverage(temp,cid)
            # Atomic replace of earlier clip ONLY after correct circle overlap and RGB QA.
            temp.replace(destination)
            results.append(dict(source_ortho=p.name,clip_file=destination.name,**diag))
            print('VERIFIED:',cid,'source:',p.name,'coverage:',round(diag['coverage_fraction']*100,2),
                  'valid pixels:',round(diag['valid_fraction']*100,2),
                  'WGS84 bounds:',diag['raster_bounds_wgs84'],flush=True)
            print('SAVED:',destination.name,'site:',p.name,'MB:',
                  round(destination.stat().st_size/1e6,2),flush=True)
        finally:
            if temp.exists():temp.unlink()
    if results:
        qa_path=OUT/'rgb_clip_spatial_qa.csv'
        incoming=pd.DataFrame(results)
        if qa_path.exists():
            old=pd.read_csv(qa_path)
            incoming=pd.concat([old[~old.circle_id.isin(target_ids)],incoming],ignore_index=True)
        incoming.sort_values('circle_id').to_csv(qa_path,index=False)
        print('SAVED spatial QA:',qa_path,flush=True)
    return results

if REBUILD_ONLY_CIRCLE_33:
    print('TARGETED REPAIR: rebuilding only circle 33 from geographically matched '
          'source ortho. Other circle TIFFs will not be touched.',flush=True)
    repair33=prepare_rgb(target_ids=[33],force_rebuild_ids={33})
    assert len(repair33)==1
    print('UPLOAD THIS FILE to replace GEE asset circle_33_UAV_RGB:',
          OUT/'circle_33_UAV_RGB.tif',flush=True)
    print('SOURCE ORTHO:',site_rgb[33].name)
    print('EXPECTED GEE WGS84 BOUNDS:',repair33[0]['raster_bounds_wgs84'])
    print('Then rerun the GEE coverage diagnostic for circle 33.',flush=True)

if RUN_EXPORT:
    assert not len(boundary_warnings) or ALLOW_BOUNDARY_WARNINGS,(
        'Circle-edge polygons require review: set ALLOW_BOUNDARY_WARNINGS=True only after inspection')
    assert not len(strong_not_exact) or ALLOW_STRONG_NONEXACT_OVERLAPS,(
        'Strong nonexact overlaps require review before export')
    prepare_vectors()
    prepare_rgb()
else:
    print('QA complete. Set RUN_EXPORT=True after reviewing circle mapping and duplicates.')


## 05 — Final manifest for manual GEE upload

In Earth Engine Asset Manager, create/use `projects/musanga-511110/assets/Verification/`. Upload **six** `circle_XX_UAV_RGB.tif` files as image assets, `verification_unique.zip` and `circles_6.zip` as table assets. The old `verification_4circles` and `circles_4` can remain unchanged.

**Do not republish the existing verification app unchanged:** it assumes 47 source rows, 39 unique candidates, and only four circle images. After this dataset is ingested, update the app to read the `verification_unique` table, use the `verify_id` field, include circles 8 and 12, and use the new canonical-ID mapping. The exact app count is determined by the duplicate audit, *not assumed in advance*.


In [ ]:
if RUN_EXPORT:
    sources=[str(site_rgb[cid].name) for cid in CIRCLE_IDS]
    manifest=dict(
        dataset='Musanga Verification v2',
        expected_source_features=EXPECTED_SOURCE_COUNT,
        actual_source_features=len(v),
        unique_features=len(canonical),
        duplicate_aliases=int(v.is_duplicate.sum()),
        circles=CIRCLE_IDS,
        circle_rgb={str(cid):site_rgb[cid].name for cid in CIRCLE_IDS},
        expected_assets=[f'circle_{cid:02d}_UAV_RGB' for cid in CIRCLE_IDS]+
                        ['verification_unique','circles_6'],
        gee_parent='projects/musanga-511110/assets/Verification',
        polygon_id_field='verify_id',
        duplicate_mapping_csv='verification_68_source_mapping.csv')
    (OUT/'verification_v2_manifest.json').write_text(json.dumps(manifest,indent=2))
    print('Ready for manual upload; unique features:',len(canonical))
    print(json.dumps(manifest,indent=2))
else:
    print('Manifest will be written after successful export.')
